# Data Leakage Check — Train/Val Similarity Analysis

DeepGlobe tiles are cut from larger satellite images. Adjacent tiles may share roads, buildings, and geographic context. If our random train/val split places adjacent tiles in different sets, the model effectively sees part of the validation data during training — inflating metrics.

### Strategy
Instead of comparing all 5292×934 pairs, we focus on the **highest IoU samples** in both sets — if leakage exists, it would inflate these scores the most.

### Methods compared
1. **CLIP embeddings** — semantic similarity (understands scene content)
2. **ResNet encoder features** — visual similarity from our trained model

---
## 1. Setup

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "open-clip-torch"], check=True)
    PROJECT_ROOT = __import__('pathlib').Path(REPO_DIR).resolve()
else:
    PROJECT_ROOT = __import__('pathlib').Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from PIL import Image

plt.style.use("ggplot")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

---
## 2. Load Train/Val Split and Identify Top-IoU Samples

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
train_pairs, val_pairs = split_pairs(pairs, val_ratio=0.15, seed=42)

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

# ====== SET YOUR CHECKPOINT (optional — for IoU-based selection) ======
CHECKPOINT = ""  # Set to best.pth path, or leave empty to use random selection

TOP_K = 100  # Number of top samples from each set to compare

In [ ]:
# If checkpoint available, select top-K by IoU; otherwise use random
if CHECKPOINT:
    print("Selecting top-K samples by IoU (running inference)...")
    from road_segmentation.models.factory import create_model
    from road_segmentation.data.transforms import get_val_transform
    from road_segmentation.data.dataset import RoadSegmentationDataset
    from torch.utils.data import DataLoader

    state = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    cfg = state.get("config", {})
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    norm_cfg = cfg.get("normalization", {})
    image_size = data_cfg.get("image_size", 512)
    mean = norm_cfg.get("mean", [0.485, 0.456, 0.406])
    std = norm_cfg.get("std", [0.229, 0.224, 0.225])

    seg_model = create_model(
        arch=model_cfg.get("arch", "Unet"),
        encoder_name=model_cfg.get("encoder_name", "resnet34"),
        encoder_weights=None,
    )
    seg_model.load_state_dict(state["model_state_dict"])
    seg_model = seg_model.to(device).eval()

    val_transform = get_val_transform(image_size, mean, std)

    def compute_ious(pairs_list, model, transform):
        ds = RoadSegmentationDataset(pairs_list, transform=transform)
        loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)
        ious = []
        with torch.no_grad():
            for batch in tqdm(loader, desc="Computing IoU"):
                imgs = batch["image"].to(device)
                masks = batch["mask"]
                probs = torch.sigmoid(model(imgs)).cpu()
                for k in range(len(probs)):
                    pred = (probs[k, 0] >= 0.5).numpy()
                    gt = (masks[k, 0].numpy() > 0)
                    tp = (pred & gt).sum()
                    union = (pred | gt).sum()
                    ious.append(tp / max(union, 1))
        return np.array(ious)

    train_ious = compute_ious(train_pairs, seg_model, val_transform)
    val_ious = compute_ious(val_pairs, seg_model, val_transform)

    train_top_idx = np.argsort(train_ious)[-TOP_K:]
    val_top_idx = np.argsort(val_ious)[-TOP_K:]

    del seg_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"Top-{TOP_K} train IoU range: [{train_ious[train_top_idx[0]]:.3f}, {train_ious[train_top_idx[-1]]:.3f}]")
    print(f"Top-{TOP_K} val IoU range:   [{val_ious[val_top_idx[0]]:.3f}, {val_ious[val_top_idx[-1]]:.3f}]")
else:
    print(f"No checkpoint — using random {TOP_K} samples from each set.")
    rng = np.random.RandomState(42)
    train_top_idx = rng.choice(len(train_pairs), TOP_K, replace=False)
    val_top_idx = rng.choice(len(val_pairs), TOP_K, replace=False)

train_selected = [train_pairs[i] for i in train_top_idx]
val_selected = [val_pairs[i] for i in val_top_idx]
print(f"\nComparing {TOP_K} train x {TOP_K} val = {TOP_K**2:,} pairs")

---
## 3. Method 1 — ResNet Feature Similarity

Extract features from a pretrained ResNet50 (ImageNet) and compute cosine similarity.

In [ ]:
import torchvision.models as models
import torchvision.transforms as T

# Load pretrained ResNet50 as feature extractor
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = torch.nn.Identity()  # Remove classification head
resnet = resnet.to(device).eval()

resnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_resnet_embeddings(pairs_list, model, transform):
    embeddings = []
    with torch.no_grad():
        for pair in tqdm(pairs_list, desc="ResNet embeddings"):
            img = Image.open(pair.image_path).convert("RGB")
            tensor = transform(img).unsqueeze(0).to(device)
            feat = model(tensor)
            feat = F.normalize(feat, dim=1)
            embeddings.append(feat.cpu())
    return torch.cat(embeddings, dim=0)


print("Extracting ResNet embeddings...")
train_resnet_emb = extract_resnet_embeddings(train_selected, resnet, resnet_transform)
val_resnet_emb = extract_resnet_embeddings(val_selected, resnet, resnet_transform)

# Cosine similarity matrix (TOP_K x TOP_K)
resnet_sim = torch.mm(val_resnet_emb, train_resnet_emb.T).numpy()
print(f"ResNet similarity matrix shape: {resnet_sim.shape}")
print(f"Max similarity: {resnet_sim.max():.4f}")
print(f"Mean similarity: {resnet_sim.mean():.4f}")

del resnet
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 4. Method 2 — CLIP Embedding Similarity

CLIP understands scene semantics — it can catch tiles from the same geographic area even with different crops or lighting.

In [ ]:
try:
    import open_clip
    CLIP_AVAILABLE = True
except ImportError:
    CLIP_AVAILABLE = False
    print("open-clip-torch not installed. Run: pip install open-clip-torch")
    print("Skipping CLIP — will use ResNet only.")

if CLIP_AVAILABLE:
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    clip_model = clip_model.to(device).eval()

    def extract_clip_embeddings(pairs_list, model, preprocess):
        embeddings = []
        with torch.no_grad():
            for pair in tqdm(pairs_list, desc="CLIP embeddings"):
                img = Image.open(pair.image_path).convert("RGB")
                tensor = preprocess(img).unsqueeze(0).to(device)
                feat = model.encode_image(tensor)
                feat = F.normalize(feat, dim=1)
                embeddings.append(feat.cpu())
        return torch.cat(embeddings, dim=0).float()

    print("Extracting CLIP embeddings...")
    train_clip_emb = extract_clip_embeddings(train_selected, clip_model, clip_preprocess)
    val_clip_emb = extract_clip_embeddings(val_selected, clip_model, clip_preprocess)

    clip_sim = torch.mm(val_clip_emb, train_clip_emb.T).numpy()
    print(f"CLIP similarity matrix shape: {clip_sim.shape}")
    print(f"Max similarity: {clip_sim.max():.4f}")
    print(f"Mean similarity: {clip_sim.mean():.4f}")

    del clip_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 5. Analysis — Similarity Distributions

In [ ]:
n_plots = 2 if CLIP_AVAILABLE else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

# ResNet distribution
resnet_flat = resnet_sim.ravel()
axes[0].hist(resnet_flat, bins=100, color="#2b8cbe", edgecolor="white", alpha=0.8)
axes[0].axvline(0.95, color="red", linestyle="--", label="Suspicious (>0.95)")
axes[0].axvline(0.90, color="orange", linestyle="--", label="High (>0.90)")
axes[0].set_title(f"ResNet Cosine Similarity (val vs train)\nmax={resnet_flat.max():.4f}")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Count")
axes[0].legend()

if CLIP_AVAILABLE:
    clip_flat = clip_sim.ravel()
    axes[1].hist(clip_flat, bins=100, color="#7bccc4", edgecolor="white", alpha=0.8)
    axes[1].axvline(0.95, color="red", linestyle="--", label="Suspicious (>0.95)")
    axes[1].axvline(0.90, color="orange", linestyle="--", label="High (>0.90)")
    axes[1].set_title(f"CLIP Cosine Similarity (val vs train)\nmax={clip_flat.max():.4f}")
    axes[1].set_xlabel("Cosine similarity")
    axes[1].legend()

plt.tight_layout()
plt.show()

# Count suspicious pairs
for thresh in [0.85, 0.90, 0.95]:
    resnet_count = (resnet_flat >= thresh).sum()
    msg = f"ResNet: {resnet_count} pairs above {thresh}"
    if CLIP_AVAILABLE:
        clip_count = (clip_flat >= thresh).sum()
        msg += f" | CLIP: {clip_count} pairs above {thresh}"
    print(msg)

---
## 6. Inspect Most Similar Pairs

Show the most suspicious val-train pairs side by side.

In [ ]:
def show_top_pairs(sim_matrix, val_list, train_list, method_name, n=10):
    """Show the n most similar val-train pairs."""
    # Get top-n pairs
    flat_idx = np.argsort(sim_matrix.ravel())[-n:][::-1]
    val_idx = flat_idx // sim_matrix.shape[1]
    train_idx = flat_idx % sim_matrix.shape[1]

    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        vi, ti = val_idx[i], train_idx[i]
        sim_score = sim_matrix[vi, ti]

        val_img = np.array(Image.open(val_list[vi].image_path).convert("RGB"))
        train_img = np.array(Image.open(train_list[ti].image_path).convert("RGB"))

        axes[i, 0].imshow(val_img)
        axes[i, 0].set_title(f"VAL: {val_list[vi].sample_id[-15:]}", fontsize=9)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(train_img)
        axes[i, 1].set_title(f"TRAIN: {train_list[ti].sample_id[-15:]}", fontsize=9)
        axes[i, 1].axis("off")

        axes[i, 0].set_ylabel(f"sim={sim_score:.3f}", fontsize=10, fontweight="bold")

    fig.suptitle(f"Top {n} Most Similar Val-Train Pairs ({method_name})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


show_top_pairs(resnet_sim, val_selected, train_selected, "ResNet", n=10)

In [ ]:
if CLIP_AVAILABLE:
    show_top_pairs(clip_sim, val_selected, train_selected, "CLIP", n=10)

---
## 7. Method Comparison — ResNet vs CLIP

In [ ]:
if CLIP_AVAILABLE:
    # For each val sample, get its max similarity to any train sample
    resnet_max_per_val = resnet_sim.max(axis=1)
    clip_max_per_val = clip_sim.max(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: ResNet max sim vs CLIP max sim per val sample
    axes[0].scatter(resnet_max_per_val, clip_max_per_val, alpha=0.5, s=20)
    axes[0].set_xlabel("ResNet max similarity")
    axes[0].set_ylabel("CLIP max similarity")
    axes[0].set_title("Per-Val-Sample: ResNet vs CLIP Max Similarity")
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)

    # Correlation
    corr = np.corrcoef(resnet_max_per_val, clip_max_per_val)[0, 1]
    axes[0].text(0.05, 0.95, f"Correlation: {corr:.3f}", transform=axes[0].transAxes,
                 fontsize=11, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Side-by-side histograms of max similarity
    axes[1].hist(resnet_max_per_val, bins=40, alpha=0.6, color="#2b8cbe", label="ResNet")
    axes[1].hist(clip_max_per_val, bins=40, alpha=0.6, color="#7bccc4", label="CLIP")
    axes[1].set_xlabel("Max similarity to any train sample")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Distribution of Nearest-Train Similarity")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"\nCorrelation between methods: {corr:.3f}")
    if corr > 0.7:
        print("High agreement — both methods flag similar pairs.")
    else:
        print("Low agreement — methods capture different aspects of similarity.")
else:
    print("CLIP not available — skipping comparison.")

---
## 8. Verdict

In [ ]:
print("Leakage Assessment")
print("=" * 60)

# Use the more sensitive method
best_sim = resnet_sim
method = "ResNet"
if CLIP_AVAILABLE and clip_sim.max() > resnet_sim.max():
    best_sim = clip_sim
    method = "CLIP"

max_sim = best_sim.max()
n_above_90 = (best_sim.ravel() >= 0.90).sum()
n_above_95 = (best_sim.ravel() >= 0.95).sum()
total_pairs = best_sim.size

print(f"Method used: {method}")
print(f"Max similarity found: {max_sim:.4f}")
print(f"Pairs above 0.90: {n_above_90}/{total_pairs} ({n_above_90/total_pairs*100:.2f}%)")
print(f"Pairs above 0.95: {n_above_95}/{total_pairs} ({n_above_95/total_pairs*100:.2f}%)")

print()
if max_sim >= 0.95:
    print("HIGH LEAKAGE RISK: Near-duplicate tiles found between train and val.")
    print("Validation metrics are likely inflated. Consider:")
    print("  1. Removing the flagged val samples and re-evaluating")
    print("  2. Reporting both inflated and cleaned metrics in the write-up")
elif max_sim >= 0.90:
    print("MODERATE LEAKAGE RISK: Some highly similar (possibly adjacent) tiles found.")
    print("Metrics may be slightly optimistic. Acknowledge in the write-up.")
elif max_sim >= 0.80:
    print("LOW LEAKAGE RISK: Some similar scenes but no near-duplicates.")
    print("This is expected for satellite imagery from similar regions.")
else:
    print("MINIMAL LEAKAGE RISK: Train and val samples appear visually distinct.")
    print("The random split is reasonably safe for this dataset.")